In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Random Forest on tradeBalance, full dataset at HTS level, no controls

## Load data, create dummy variables, scale data, train/test split

In [2]:
data = pd.read_csv("MLReadyCombinedDataset.csv",usecols = ["Country","Year","HTS8","tradeBalance","mfn_ad_val_rate"] )

C:\Users\iiski\AppData\Local\Temp\ipykernel_2848\4169333363.py:1: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv("MLReadyCombinedDataset.csv",usecols = ["Country","Year","HTS8","tradeBalance","mfn_ad_val_rate"] )


In [3]:
data = data.dropna(subset = ['tradeBalance','Year',"mfn_ad_val_rate"])

In [4]:
#Modeled after the example here: https://pandas.pydata.org/docs/user_guide/groupby.html
data['tradeBalance'] = data.groupby(['Country','HTS8'])['tradeBalance'].transform(lambda x: (x - x.mean()) / x.std())

In [5]:
#Drop NAs again because scaling introduces NAs
data = data.dropna(subset = ['tradeBalance'])

In [6]:
data = pd.get_dummies(data,columns = ["Country"])

In [7]:
y = data['tradeBalance']
X = data.drop(['tradeBalance'],axis=1)

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42)

## Fitting the Random Forest and calculating RMSE

In [9]:
rf = RandomForestRegressor(random_state=42,max_depth = 5,verbose = 1)
rf.fit(X_train,y_train)
y_pred = rf.predict(X_test)

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done 100 out of 100 | elapsed: 127.0min finished
[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done 100 out of 100 | elapsed:   11.1s finished


In [10]:
forestMSE = ((y_pred-y_test)**2).sum()/len(y_pred)
forestMSE

0.9192172723066997

In [11]:
forestRMSE = forestMSE**.5
forestRMSE

0.9587581928237692

In [12]:
feature_importances = pd.DataFrame({
    "feature": X.columns,
    "importance": rf.feature_importances_
}).sort_values(by="importance", ascending=False)

In [13]:
feature_importances.head(20)

,feature,importance
0,Year,0.627504
44,Country_China,0.120713
2,mfn_ad_val_rate,0.083059
228,Country_Venezuela,0.062924
98,Country_India,0.039840
1,HTS8,0.033760
222,Country_United Arab Emirates,0.022280
10,Country_Argentina,0.004589
29,Country_Brazil,0.003252
81,Country_Germany,0.001337


# Attempt at Random Forest with all variables (tariff exemption indicators, etc.)

I recommend restarting the kernel/clearing the environment here due to memory allocation issues.

## Load data, create dummy variables, establish X and y

In [2]:
data = pd.read_csv("MLReadyCombinedDataset.csv",usecols = list(range(1,35)))

C:\Users\iiski\AppData\Local\Temp\ipykernel_19352\2946809266.py:1: DtypeWarning: Columns (2,26,29) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv("MLReadyCombinedDataset.csv",usecols = list(range(1,35)))


In [3]:
data = data.dropna(subset = ['tradeBalance','Year',"mfn_ad_val_rate"])

In [4]:
data

,Year,Country,HTS8,tradeBalance,Exports,Imports,quantity_1_code,quantity_2_code,wto_binding_code,mfn_rate_type_code,...,israel_fta_indicator,atpa_indicator,pharmaceutical_ind,dyes_indicator,col2_rate_type_code,col2_ad_val_rate,col2_specific_rate,col2_other_rate,begin_effect_date,end_effective_date
0,1997.0,Afghanistan,2013060.0,62066.0,62066.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1997.0,Afghanistan,7051100.0,10615.0,10615.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1997.0,Afghanistan,10011000.0,2290359.0,2290359.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1997.0,Afghanistan,17026000.0,15983.0,15983.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1997.0,Afghanistan,20099040.0,11041.0,11041.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14617557,2025.0,Zimbabwe,97031000.0,-2250.0,0.0,2250.0,NO,NaN,B,0.0,...,NaN,NaN,NaN,NaN,0,0.00,0.00,0.00,2022-01-27,2050-12-31
14617558,2025.0,Zimbabwe,97039000.0,-308843.0,0.0,308843.0,NO,NaN,B,0.0,...,NaN,NaN,NaN,NaN,0,0.00,0.00,0.00,2022-01-27,2050-12-31
14617559,2025.0,Zimbabwe,97051000.0,-8000.0,0.0,8000.0,NaN,NaN,B,0.0,...,NaN,NaN,NaN,NaN,0,0.00,0.00,0.00,2022-01-27,2050-12-31
14617560,2025.0,Zimbabwe,98010010.0,-170086.0,0.0,170086.0,NaN,NaN,U,0.0,...,NaN,NaN,NaN,NaN,N,9999.99,9999.99,9999.99,1993-08-01,2050-12-31


In [5]:
data.columns

Index(['Year', 'Country', 'HTS8', 'tradeBalance', 'Exports', 'Imports',
       'quantity_1_code', 'quantity_2_code', 'wto_binding_code',
       'mfn_rate_type_code', 'mfn_ad_val_rate', 'mfn_specific_rate',
       'mfn_other_rate', 'gsp_indicator', 'gsp_ctry_excluded',
       'apta_indicator', 'civil_air_indicator', 'nafta_canada_ind',
       'nafta_mexico_ind', 'mexico_ad_val_rate', 'mexico_specific_rate',
       'cbi_indicator', 'cbi_ad_val_rate', 'cbi_specific_rate',
       'israel_fta_indicator', 'atpa_indicator', 'pharmaceutical_ind',
       'dyes_indicator', 'col2_rate_type_code', 'col2_ad_val_rate',
       'col2_specific_rate', 'col2_other_rate', 'begin_effect_date',
       'end_effective_date'],
      dtype='object')

In [6]:
catCols = ['Country','quantity_1_code', 'quantity_2_code', 'wto_binding_code',
       'mfn_rate_type_code','gsp_indicator', 'gsp_ctry_excluded',
       'apta_indicator', 'nafta_canada_ind',
       'nafta_mexico_ind','cbi_indicator','israel_fta_indicator', 'atpa_indicator', 'col2_rate_type_code','dyes_indicator','pharmaceutical_ind','civil_air_indicator']

In [7]:
#Dropping a few more variables. Rates and dates have frequent NAs. We will instead focus on countries and codes representing different exemptions.
data = data.drop(["mfn_specific_rate","mfn_other_rate","mexico_ad_val_rate","mexico_specific_rate","cbi_ad_val_rate","cbi_specific_rate","col2_ad_val_rate","col2_specific_rate","col2_other_rate",'begin_effect_date','end_effective_date'],axis=1)

In [8]:
data['tradeBalance'] = data.groupby(['Country','HTS8'])['tradeBalance'].transform(lambda x: (x - x.mean()) / x.std())

In [9]:
data

,Year,Country,HTS8,tradeBalance,Exports,Imports,quantity_1_code,quantity_2_code,wto_binding_code,mfn_rate_type_code,...,apta_indicator,civil_air_indicator,nafta_canada_ind,nafta_mexico_ind,cbi_indicator,israel_fta_indicator,atpa_indicator,pharmaceutical_ind,dyes_indicator,col2_rate_type_code
0,1997.0,Afghanistan,2013060.0,NaN,62066.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1997.0,Afghanistan,7051100.0,NaN,10615.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1997.0,Afghanistan,10011000.0,0.707107,2290359.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1997.0,Afghanistan,17026000.0,NaN,15983.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1997.0,Afghanistan,20099040.0,NaN,11041.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14617557,2025.0,Zimbabwe,97031000.0,0.656458,0.0,2250.0,NO,NaN,B,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
14617558,2025.0,Zimbabwe,97039000.0,0.604528,0.0,308843.0,NO,NaN,B,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
14617559,2025.0,Zimbabwe,97051000.0,0.722410,0.0,8000.0,NaN,NaN,B,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
14617560,2025.0,Zimbabwe,98010010.0,0.491634,0.0,170086.0,NaN,NaN,U,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N


In [10]:
data = pd.get_dummies(data,columns = catCols, dummy_na = True)
#data = pd.get_dummies(data,columns = ["Country"])

In [11]:
sample = data.sample(frac=0.15,random_state = 42)

In [12]:
sample = sample.dropna(subset = ['tradeBalance'])

In [13]:
sample

,Year,HTS8,tradeBalance,Exports,Imports,mfn_ad_val_rate,Country_Afghanistan,Country_Albania,Country_Algeria,Country_Andorra,...,col2_rate_type_code_7,col2_rate_type_code_9,col2_rate_type_code_?,col2_rate_type_code_N,col2_rate_type_code_T,col2_rate_type_code_X,col2_rate_type_code_nan,dyes_indicator_nan,pharmaceutical_ind_nan,civil_air_indicator_nan
5747646,2013.0,72261100.0,-0.680629,28189.0,0.0,0.000,False,False,False,False,...,False,False,False,False,False,False,True,True,True,True
5031269,2011.0,74152900.0,0.673912,55333.0,0.0,0.030,False,False,False,False,...,True,False,False,False,False,False,False,True,True,True
10018855,2025.0,68138900.0,-0.557470,10668394.0,10399890.0,0.000,False,False,False,False,...,True,False,False,False,False,False,False,True,True,True
13176176,2017.0,84209920.0,0.065828,0.0,73842.0,0.000,False,False,False,False,...,True,False,False,False,False,False,False,True,True,True
3006178,2006.0,84829100.0,2.341191,690654.0,0.0,0.044,False,False,False,False,...,True,False,False,False,False,False,False,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9020905,2022.0,62014020.0,-0.795719,13576.0,595737.0,0.277,False,False,False,False,...,True,False,False,False,False,False,False,True,True,True
266863,1997.0,90149000.0,0.596605,55586.0,0.0,0.000,False,False,False,False,...,False,False,False,False,False,False,True,True,True,True
6547673,2015.0,48192000.0,-0.032687,90657.0,4183.0,0.000,False,False,False,False,...,False,False,False,False,False,False,False,True,True,True
5300959,2011.0,12040000.0,0.715276,116077.0,0.0,0.000,False,False,False,False,...,False,False,False,False,False,False,True,True,True,True


In [14]:
y = sample['tradeBalance']
X = sample.drop(['tradeBalance','Exports','Imports'],axis=1)

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42)

In [16]:
dt = DecisionTreeRegressor(random_state=42,max_depth = 5)
dt.fit(X_train,y_train)
y_pred = dt.predict(X_test)

In [17]:
treeMSE = ((y_pred-y_test)**2).sum()/len(y_pred)
treeMSE

0.9184816713728581

In [18]:
treeRMSE = treeMSE**.5
treeRMSE

0.9583744943250827

In [19]:
feature_importances = pd.DataFrame({
    "feature": X.columns,
    "importance": dt.feature_importances_
}).sort_values(by="importance", ascending=False)

In [20]:
feature_importances.head(20)

,feature,importance
0,Year,0.502216
302,mfn_rate_type_code_nan,0.198675
44,Country_China,0.090744
98,Country_India,0.066908
228,Country_Venezuela,0.053681
295,wto_binding_code_nan,0.035549
243,quantity_1_code_DOZ,0.023096
229,Country_Vietnam,0.012898
1,HTS8,0.009653
106,Country_Japan,0.004437


In [21]:
rf = RandomForestRegressor(random_state=42,max_depth = 5,verbose = 1)
rf.fit(X_train,y_train)
y_pred = rf.predict(X_test)

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done 100 out of 100 | elapsed: 23.9min finished
[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done 100 out of 100 | elapsed:    2.0s finished


In [22]:
forestMSE = ((y_pred-y_test)**2).sum()/len(y_pred)
forestMSE

0.917984508345988

In [23]:
forestRMSE = forestMSE**.5
forestRMSE

0.9581150809511287

In [24]:
feature_importances = pd.DataFrame({
    "feature": X.columns,
    "importance": rf.feature_importances_
}).sort_values(by="importance", ascending=False)

In [25]:
feature_importances.head(20)

,feature,importance
0,Year,0.509923
44,Country_China,0.091670
295,wto_binding_code_nan,0.088459
302,mfn_rate_type_code_nan,0.081753
98,Country_India,0.056662
477,col2_rate_type_code_nan,0.051933
228,Country_Venezuela,0.041248
243,quantity_1_code_DOZ,0.021439
229,Country_Vietnam,0.019594
1,HTS8,0.016775
